# AIC2024 KIS — Phase 3: caption + OCR bằng Vintern-1B-v3.5 (Kaggle)

Notebook **self-contained**: đọc bundle keyframe (`manifest.jsonl` + `keyframes/`), chạy
`5CD-AI/Vintern-1B-v3_5` trên từng keyframe với một prompt duy nhất trả về JSON
`{"image-captioning", "ocr"}`, rồi xuất **shard JSONL theo video**.

## Lưu gì?

Artifact gốc là JSONL, một dòng một keyframe (giống object shard):

```
caption_ocr_shards/
  shards/<video_id>/caption_ocr.jsonl
  shards/<video_id>/manifest.json
```

```json
{"kf_uid": 1, "video_id": "L01_V001", "generation": 0, "shot_idx": 0, "kf_idx": 0, "ts_ms": 2640, "caption": "Một nam phát thanh viên...", "ocr": "THỜI SỰ 19H", "parsed": true}
```

- `caption`: mô tả chi tiết nội dung ảnh.
- `ocr`: toàn bộ chữ trích xuất được, `""` nếu không có.
- `parsed`: `false` khi model không trả JSON hợp lệ; lúc đó có thêm `raw` để xử lý lại sau.
- Notebook **chỉ lưu JSON**, chưa ghi gì vào SQLite/FAISS; import quyết định sau.

## Chuẩn bị

1. Ở local tạo bundle: `uv run ir export-keyframes` → `artifacts_keyframes.zip`.
2. Upload zip đó thành Kaggle Dataset.
3. Notebook Settings: bật **GPU** (T4/P100 đủ) và **Internet** (tải weights).
4. Add dataset, chọn batch (bên dưới), chạy hết cell.
5. Tải `artifacts_caption_ocr_shards.zip` về máy.

## Chọn batch

- Mặc định: tất cả video trong manifest.
- Theo lô: `VINTERN_VIDEO_OFFSET` + `VINTERN_VIDEO_LIMIT` (ví dụ 0/100, 100/100, ...).
- Hoặc chỉ định: `VINTERN_VIDEO_IDS=L01_V001,L01_V002`.
- Resume: shard đã xong trong `WORK_DIR` được bỏ qua; shard dở dang chạy tiếp từ dòng cuối.

## Chi phí

280k keyframe × ~200 token sinh ra là bài toán nhiều giờ GPU. Muốn thử trước thì đặt
`VINTERN_VIDEO_LIMIT=1` rồi xem thời gian mỗi video ở cell 7; hoặc giảm `VINTERN_MAX_NUM`
(1–2 patch) để nhanh hơn nhưng OCR chữ nhỏ kém hơn.


## 1. Cấu hình

In [ ]:
import os
from pathlib import Path

print("START: cấu hình", flush=True)

MODEL_ID = os.environ.get("VINTERN_MODEL", "5CD-AI/Vintern-1B-v3_5")
IMAGE_SIZE = int(os.environ.get("VINTERN_IMAGE_SIZE", "448"))
MAX_NUM = int(os.environ.get("VINTERN_MAX_NUM", "6"))
USE_THUMBNAIL = os.environ.get("VINTERN_USE_THUMBNAIL", "1") != "0"

BATCH_SIZE = int(os.environ.get("VINTERN_BATCH_SIZE", "2"))
NUM_WORKERS = int(os.environ.get("VINTERN_NUM_WORKERS", "4"))
PROGRESS_EVERY = int(os.environ.get("VINTERN_PROGRESS_EVERY", "20"))

MAX_NEW_TOKENS = int(os.environ.get("VINTERN_MAX_NEW_TOKENS", "1024"))
NUM_BEAMS = int(os.environ.get("VINTERN_NUM_BEAMS", "1"))
REPETITION_PENALTY = float(os.environ.get("VINTERN_REPETITION_PENALTY", "1.1"))
PROMPT_VERSION = os.environ.get("VINTERN_PROMPT_VERSION", "caption_ocr_vi_v1")

BUNDLE_DIR = os.environ.get("VINTERN_BUNDLE_DIR") or None
WORK_DIR = Path(os.environ.get("VINTERN_WORK_DIR", "/kaggle/working/caption-ocr-vintern"))
SHARDS_DIR = Path(os.environ.get("VINTERN_SHARDS_DIR", str(WORK_DIR / "shards")))
ZIP_PATH = Path(
    os.environ.get(
        "VINTERN_ZIP_PATH",
        "/kaggle/working/artifacts_caption_ocr_shards.zip",
    )
)
RESUME = os.environ.get("VINTERN_RESUME", "1") != "0"
REQUIRE_GPU = os.environ.get("VINTERN_REQUIRE_GPU", "1") != "0"
SMOKE_TEST = os.environ.get("VINTERN_SMOKE", "1") != "0"
DTYPE_NAME = os.environ.get("VINTERN_DTYPE") or None

VIDEO_IDS = [
    value.strip()
    for value in os.environ.get("VINTERN_VIDEO_IDS", "").split(",")
    if value.strip()
]
VIDEO_OFFSET = int(os.environ.get("VINTERN_VIDEO_OFFSET", "0"))
VIDEO_LIMIT = int(os.environ.get("VINTERN_VIDEO_LIMIT", "0"))

print("model:", MODEL_ID, flush=True)
print(
    f"image_size: {IMAGE_SIZE} | max_num: {MAX_NUM} | thumbnail: {USE_THUMBNAIL}",
    flush=True,
)
print(
    f"batch: {BATCH_SIZE} | decode threads: {NUM_WORKERS} | "
    f"progress every: {PROGRESS_EVERY} batches",
    flush=True,
)
print(
    f"generation: max_new_tokens={MAX_NEW_TOKENS} beams={NUM_BEAMS} "
    f"repetition_penalty={REPETITION_PENALTY}",
    flush=True,
)
print("prompt:", PROMPT_VERSION, "| dtype:", DTYPE_NAME or "auto", flush=True)
print("bundle override:", BUNDLE_DIR, flush=True)
print("work:", WORK_DIR, flush=True)
print("shards:", SHARDS_DIR, flush=True)
print("zip:", ZIP_PATH, flush=True)
print(
    "video_ids:", VIDEO_IDS or "(all)",
    "| offset:", VIDEO_OFFSET,
    "| limit:", VIDEO_LIMIT,
    flush=True,
)


## 2. Cài dependency

Kaggle đã có `torch`/`torchvision`; cell này ghim bộ version mà remote code của
InternVL/Vintern cần (`transformers==4.46.3`, `tokenizers==0.20.3`, `timm==1.0.9`).

- Nếu muốn dùng bản có sẵn của Kaggle: `VINTERN_SKIP_INSTALL=1`.
- Nếu pin bị xung đột: `VINTERN_PACKAGES="einops addict"` để chỉ cài phần thiếu.
- `flash-attn` **không** cài; notebook dùng `use_flash_attn=False`.


In [ ]:
import os
import subprocess
import sys

print("python:", sys.version.split()[0], flush=True)

DEFAULT_PACKAGES = (
    "transformers==4.46.3 tokenizers==0.20.3 timm==1.0.9 "
    "einops addict accelerate sentencepiece"
)
PACKAGES = [
    value
    for value in os.environ.get("VINTERN_PACKAGES", DEFAULT_PACKAGES).split()
    if value
]

if os.environ.get("VINTERN_SKIP_INSTALL") == "1":
    print("skip install (VINTERN_SKIP_INSTALL=1)", flush=True)
elif not PACKAGES:
    print("skip install (VINTERN_PACKAGES rỗng)", flush=True)
else:
    print("installing:", " ".join(PACKAGES), flush=True)
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *PACKAGES],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(result.stdout[-4000:], flush=True)
        print(result.stderr[-4000:], flush=True)
        raise RuntimeError(
            "pip install lỗi — đặt VINTERN_PACKAGES để chỉ cài phần còn thiếu, "
            "hoặc VINTERN_SKIP_INSTALL=1 để dùng bản có sẵn của Kaggle."
        )
    print("install done", flush=True)


## 3. Tìm bundle keyframe

Tìm `manifest.jsonl` có thư mục `keyframes/` đi kèm trong `/kaggle/input`.
Manifest chứa `kf_uid`, đường dẫn ảnh và metadata; không cần đọc SQLite.


In [ ]:
import json

candidates = []
seen = set()


def add_bundle(candidate: Path) -> None:
    bundle = candidate.resolve()
    if bundle in seen:
        return
    seen.add(bundle)
    if (bundle / "manifest.jsonl").is_file() and (bundle / "keyframes").is_dir():
        candidates.append(bundle)


if BUNDLE_DIR:
    # Đường dẫn đã chỉ định thì dùng thẳng. rglob trên /kaggle/input (FUSE mount,
    # 280k file) tốn nhiều phút mà không tìm được bundle nào khác.
    add_bundle(Path(BUNDLE_DIR))
else:
    for root in (Path("/kaggle/input"), Path.cwd()):
        if not root.is_dir():
            continue
        for manifest in sorted(root.rglob("manifest.jsonl")):
            add_bundle(manifest.parent)

if not candidates:
    raise FileNotFoundError(
        "Không tìm thấy manifest.jsonl + keyframes/. "
        "Set VINTERN_BUNDLE_DIR hoặc kiểm tra lại Kaggle dataset."
    )

BUNDLE_DIR = candidates[0]
MANIFEST_PATH = BUNDLE_DIR / "manifest.jsonl"

rows = []
for line in MANIFEST_PATH.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if line:
        rows.append(json.loads(line))
if not rows:
    raise RuntimeError("manifest.jsonl rỗng")

print("bundle:", BUNDLE_DIR)
print("keyframes:", BUNDLE_DIR / "keyframes")
print("manifest rows:", f"{len(rows):,}")
if len(candidates) > 1:
    print("bỏ qua bundle khác:", [str(path) for path in candidates[1:]])


## 4. Chọn batch video

Thứ tự video lấy theo `video_id` đã sort. `VINTERN_VIDEO_OFFSET`/`VINTERN_VIDEO_LIMIT`
cho phép chạy lô 100 video mỗi lần mà không cần sửa notebook.


In [ ]:
by_video: dict[str, list[dict]] = {}
for row in sorted(rows, key=lambda item: (str(item["video_id"]), int(item["kf_uid"]))):
    by_video.setdefault(str(row["video_id"]), []).append(row)

all_video_ids = sorted(by_video)
if VIDEO_IDS:
    selected = list(dict.fromkeys(VIDEO_IDS))
    missing_videos = [video_id for video_id in selected if video_id not in by_video]
    if missing_videos:
        raise ValueError(f"VINTERN_VIDEO_IDS không có trong manifest: {missing_videos}")
elif VIDEO_LIMIT > 0:
    selected = all_video_ids[VIDEO_OFFSET : VIDEO_OFFSET + VIDEO_LIMIT]
else:
    selected = all_video_ids[VIDEO_OFFSET:]

if not selected:
    raise RuntimeError("Không có video nào được chọn.")

total_selected = sum(len(by_video[video_id]) for video_id in selected)
print(f"videos in manifest: {len(all_video_ids):,}")
print(f"selected videos: {len(selected):,} | keyframes: {total_selected:,}")
print(f"batch: offset={VIDEO_OFFSET} limit={VIDEO_LIMIT} explicit={len(VIDEO_IDS)}")
if selected:
    print("first:", selected[0], "| last:", selected[-1])
if not VIDEO_IDS and VIDEO_LIMIT > 0:
    print("next VINTERN_VIDEO_OFFSET:", VIDEO_OFFSET + len(selected))


## 5. Prompt và parse JSON

Model được yêu cầu trả về **một** object JSON. Thực tế model hay bọc trong ```json fence
hoặc thêm chữ phía trước, nên `parse_response` bóc fence, cắt từ `{` đầu tới `}` cuối, đọc
các tên trường tương đương (`image-captioning` / `image_captioning` / `caption`) và trả
`parsed=False` khi bó tay — dòng đó vẫn được lưu (kèm `raw`) để xử lý lại sau.


In [ ]:
import json
import re

PROMPT = (
    "<image>\n"
    "Đối với hình ảnh được cung cấp, hãy thực hiện hai tác vụ sau và trả về kết quả dưới dạng một đối tượng JSON duy nhất:\n"
    "1.  **image-captioning**: Cung cấp mô tả chi tiết về nội dung của hình ảnh.\n"
    "2.  **ocr**: Trích xuất toàn bộ văn bản (chữ) có thể nhận diện được từ hình ảnh. Nếu không có văn bản nào được nhận diện, hãy trả về \"\".\n"
    "Đảm bảo rằng văn bản được trích xuất giữ nguyên định dạng và cấu trúc ban đầu.\n"
    "Định dạng phản hồi phải là một đối tượng JSON với hai trường: \"image-captioning\" và \"ocr\".\n"
)

FENCE_RE = re.compile(r"```(?:json|JSON)?\s*(.*?)\s*```", re.DOTALL)
CAPTION_KEYS = (
    "image-captioning",
    "image_captioning",
    "image-caption",
    "caption",
    "description",
)
OCR_KEYS = ("ocr", "ocr-text", "ocr_text", "text", "extracted_text")


def _as_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, str):
        return value.strip()
    return json.dumps(value, ensure_ascii=False).strip()


def _first_text(data: dict, keys: tuple[str, ...]) -> str:
    for key in keys:
        if key not in data:
            continue
        value = data[key]
        if isinstance(value, dict):
            inner = _first_text(value, ("text", "content", "value", "result"))
            if inner:
                return inner
            continue
        text = _as_text(value)
        if text:
            return text
    return ""


def load_json_object(text: str) -> dict | None:
    body = text.strip()
    fence = FENCE_RE.search(body)
    if fence:
        body = fence.group(1).strip()
    candidates = [body]
    start, end = body.find("{"), body.rfind("}")
    if 0 <= start < end:
        candidates.append(body[start : end + 1])
    for candidate in candidates:
        if not candidate:
            continue
        try:
            data = json.loads(candidate)
        except ValueError:
            continue
        if isinstance(data, dict):
            return data
    return None


def parse_response(text: str) -> dict:
    data = load_json_object(text)
    if data is None:
        return {"caption": "", "ocr": "", "parsed": False}
    caption = _first_text(data, CAPTION_KEYS)
    ocr = _first_text(data, OCR_KEYS)
    return {"caption": caption, "ocr": ocr, "parsed": bool(caption or ocr)}


print("prompt:", PROMPT_VERSION, flush=True)
print("----")
print(PROMPT, flush=True)
print("----")

sample = '```json\n{"image-captioning": "Một nam phát thanh viên ngồi trước màn hình xanh.", "ocr": "THỜI SỰ 19H\\nNgày 12/10/2024"}\n```'
print("self-test:", json.dumps(parse_response(sample), ensure_ascii=False), flush=True)
assert parse_response(sample)["parsed"] is True
assert parse_response(sample)["ocr"].startswith("THỜI SỰ")
assert parse_response("xin chào")["parsed"] is False


## 6. Load Vintern-1B-v3.5

- **T4/P100 không có bf16**, mà fp16 với Qwen2 hay sinh NaN/lặp từ → notebook mặc định
  `float32` trên GPU cũ, `bfloat16` trên L4/A100. Override bằng
  `VINTERN_DTYPE=float16|bfloat16|float32` nếu muốn đổi (fp16 nhanh hơn nhưng nên soi
  `parsed`/mẫu trước khi chạy cả batch).
- `use_flash_attn=False` (không cài flash-attn), `trust_remote_code=True` để nạp
  `modeling_internvl_chat` từ HF.
- `VINTERN_MAX_NUM` là số patch tối đa mỗi ảnh (`max_dynamic_patch` của model là 4 nhưng
  có thể ép lên 6 cho ảnh nhiều chữ; càng nhiều patch càng chậm và tốn VRAM).


In [ ]:
import sys

print("python:", sys.version.split()[0], flush=True)

import torch
import torchvision
import transformers
from PIL import Image
from torchvision.transforms import functional as TF
from transformers import AutoModel, AutoTokenizer

print("torch:", torch.__version__, flush=True)
print("torchvision:", torchvision.__version__, flush=True)
print("transformers:", transformers.__version__, flush=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    capability = torch.cuda.get_device_capability(0)
    # T4/P100 không có bf16; fp16 với Qwen2 hay sinh NaN/lặp từ, nên mặc định fp32.
    auto_dtype = "bfloat16" if capability[0] >= 8 else "float32"
    print(
        "device:", torch.cuda.get_device_name(0),
        "| capability:", capability,
        "| auto dtype:", auto_dtype,
        flush=True,
    )
else:
    auto_dtype = "float32"
    print("device: cpu", flush=True)
    if REQUIRE_GPU:
        raise RuntimeError(
            "GPU không khả dụng — bật Accelerator=GPU trong Kaggle Settings "
            "(T4 x2 hoặc P100). Đặt VINTERN_REQUIRE_GPU=0 để chạy CPU (rất chậm)."
        )

DTYPE_NAME = DTYPE_NAME or auto_dtype
MODEL_DTYPE = getattr(torch, DTYPE_NAME)
print("model dtype:", DTYPE_NAME, flush=True)

print("loading model:", MODEL_ID, "(lần đầu sẽ tải weights)", flush=True)
model = (
    AutoModel.from_pretrained(
        MODEL_ID,
        torch_dtype=MODEL_DTYPE,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        use_flash_attn=False,
    )
    .eval()
    .to(DEVICE)
)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=False,
)
print("model loaded", flush=True)
print(
    "dynamic_image_size:", getattr(model.config, "dynamic_image_size", None),
    "| max_dynamic_patch:", getattr(model.config, "max_dynamic_patch", None),
    "| force_image_size:", getattr(model.config, "force_image_size", None),
    flush=True,
)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float("inf")
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio


def dynamic_preprocess(
    image: Image.Image,
    min_num: int = 1,
    max_num: int = 12,
    image_size: int = 448,
    use_thumbnail: bool = False,
) -> list[Image.Image]:
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height
    target_ratios = sorted(
        {
            (i, j)
            for n in range(min_num, max_num + 1)
            for i in range(1, n + 1)
            for j in range(1, n + 1)
            if min_num <= i * j <= max_num
        },
        key=lambda item: item[0] * item[1],
    )
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size
    )
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    resized_img = image.resize((target_width, target_height))
    processed = []
    for index in range(blocks):
        box = (
            (index % (target_width // image_size)) * image_size,
            (index // (target_width // image_size)) * image_size,
            ((index % (target_width // image_size)) + 1) * image_size,
            ((index // (target_width // image_size)) + 1) * image_size,
        )
        processed.append(resized_img.crop(box))
    if use_thumbnail and len(processed) != 1:
        processed.append(image.resize((image_size, image_size)))
    return processed


def preprocess_crop(crop: Image.Image) -> torch.Tensor:
    if crop.mode != "RGB":
        crop = crop.convert("RGB")
    resized = crop.resize(
        (IMAGE_SIZE, IMAGE_SIZE), resample=Image.Resampling.BICUBIC
    )
    tensor = TF.to_tensor(resized)
    return TF.normalize(tensor, IMAGENET_MEAN, IMAGENET_STD)


def load_image(path: Path) -> torch.Tensor:
    with Image.open(path) as image:
        rgb = image.convert("RGB")
    crops = dynamic_preprocess(
        rgb,
        image_size=IMAGE_SIZE,
        max_num=MAX_NUM,
        use_thumbnail=USE_THUMBNAIL,
    )
    return torch.stack([preprocess_crop(crop) for crop in crops])


GENERATION_CONFIG = {
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": False,
    "num_beams": NUM_BEAMS,
    "repetition_penalty": REPETITION_PENALTY,
}


def _is_oom(exc: BaseException) -> bool:
    return isinstance(exc, torch.cuda.OutOfMemoryError) or "out of memory" in str(
        exc
    ).lower()


def _infer_batch(tensors: list[torch.Tensor]) -> list[str]:
    pixel_values = torch.cat(tensors).to(DEVICE, dtype=MODEL_DTYPE)
    num_patches_list = [int(tensor.shape[0]) for tensor in tensors]
    with torch.inference_mode():
        responses = model.batch_chat(
            tokenizer,
            pixel_values,
            [PROMPT] * len(tensors),
            dict(GENERATION_CONFIG),
            num_patches_list=num_patches_list,
        )
    del pixel_values
    return [str(response) for response in responses]


def infer_batch(tensors: list[torch.Tensor]) -> list[str]:
    try:
        return _infer_batch(tensors)
    except (torch.cuda.OutOfMemoryError, RuntimeError) as exc:
        if not _is_oom(exc):
            raise
        torch.cuda.empty_cache()
        print(
            f"OOM với batch {len(tensors)} — chạy lại từng ảnh",
            flush=True,
        )
    responses = []
    for tensor in tensors:
        try:
            responses.extend(_infer_batch([tensor]))
        except (torch.cuda.OutOfMemoryError, RuntimeError) as exc:
            if not _is_oom(exc):
                raise
            torch.cuda.empty_cache()
            raise RuntimeError(
                "Hết VRAM ngay cả với 1 ảnh — giảm VINTERN_MAX_NUM (6 -> 4 -> 2) "
                "rồi chạy lại; shard sẽ resume từ keyframe vừa xong."
            ) from exc
    return responses


## 7. Smoke test 1 keyframe

Chạy trước khi quét cả batch để thấy model trả JSON đúng chưa (mất vài giây).
Tắt bằng `VINTERN_SMOKE=0`.


In [ ]:
if SMOKE_TEST:
    probe_row = by_video[selected[0]][0]
    probe_tensor = load_image(BUNDLE_DIR / probe_row["path"])
    print(
        "keyframe:", probe_row["path"],
        "| patches:", int(probe_tensor.shape[0]),
        flush=True,
    )
    raw = infer_batch([probe_tensor])[0]
    print("\n--- raw ---\n" + raw, flush=True)
    print(
        "\n--- parsed ---\n"
        + json.dumps(parse_response(raw), ensure_ascii=False, indent=2),
        flush=True,
    )
    del probe_tensor
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
else:
    print("skip smoke test (VINTERN_SMOKE=0)", flush=True)


## 8. Sinh caption + OCR từng video thành shard

Mỗi video ghi vào `SHARDS_DIR/<video_id>/caption_ocr.jsonl`. Shard đã xong được bỏ qua khi
resume; shard dở dang được cắt về đúng số dòng hợp lệ rồi chạy tiếp.

`plan_hash` gồm model + prompt + tham số sinh + danh sách `kf_uid`, nên đổi prompt hay đổi
`MAX_NUM`/`MAX_NEW_TOKENS` sẽ tự làm mới shard thay vì trộn kết quả cũ.


In [ ]:
import hashlib
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import UTC, datetime


def write_json(path: Path, payload: dict) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)


def make_record(row: dict, response: str) -> dict:
    parsed = parse_response(response)
    record = {
        "kf_uid": int(row["kf_uid"]),
        "video_id": str(row["video_id"]),
        "generation": int(row.get("generation", 0)),
        "shot_idx": int(row["shot_idx"]),
        "kf_idx": int(row["kf_idx"]),
        "ts_ms": int(row["ts_ms"]),
        "caption": parsed["caption"],
        "ocr": parsed["ocr"],
        "parsed": parsed["parsed"],
    }
    if not parsed["parsed"]:
        record["raw"] = response.strip()[:4000]
    return record


def shard_manifest(
    video_id: str,
    total: int,
    plan_hash: str,
    caption_nonempty: int,
    ocr_nonempty: int,
    parse_failed: int,
) -> dict:
    return {
        "version": 1,
        "video_id": video_id,
        "created_at": datetime.now(UTC).isoformat(),
        "model": MODEL_ID,
        "prompt_version": PROMPT_VERSION,
        "image_size": IMAGE_SIZE,
        "max_num": MAX_NUM,
        "use_thumbnail": USE_THUMBNAIL,
        "max_new_tokens": MAX_NEW_TOKENS,
        "num_beams": NUM_BEAMS,
        "repetition_penalty": REPETITION_PENALTY,
        "dtype": DTYPE_NAME,
        "count": total,
        "plan_hash": plan_hash,
        "caption_nonempty": caption_nonempty,
        "ocr_nonempty": ocr_nonempty,
        "parse_failed": parse_failed,
        "files": {"caption_ocr": "caption_ocr.jsonl"},
    }


def uids_of(rows: list[dict]) -> list[int]:
    return [int(row["kf_uid"]) for row in rows]


def plan_hash_of(video_rows: list[dict]) -> str:
    payload = "|".join(
        [
            MODEL_ID,
            PROMPT_VERSION,
            str(IMAGE_SIZE),
            str(MAX_NUM),
            str(int(USE_THUMBNAIL)),
            str(MAX_NEW_TOKENS),
            str(NUM_BEAMS),
            str(REPETITION_PENALTY),
            ",".join(map(str, uids_of(video_rows))),
        ]
    )
    return hashlib.sha256(payload.encode()).hexdigest()


decode_pool = ThreadPoolExecutor(max_workers=NUM_WORKERS)
shard_summaries = []
overall_started = time.time()
done_keyframes = 0
print(f"generating {len(selected):,} videos -> {SHARDS_DIR}", flush=True)

for position, video_id in enumerate(selected, start=1):
    video_rows = by_video[video_id]
    total = len(video_rows)
    uids = uids_of(video_rows)
    current_hash = plan_hash_of(video_rows)

    shard_dir = SHARDS_DIR / video_id
    shard_dir.mkdir(parents=True, exist_ok=True)
    records_path = shard_dir / "caption_ocr.jsonl"
    manifest_path = shard_dir / "manifest.json"
    plan_path = shard_dir / "plan.json"
    progress_path = shard_dir / "progress.json"

    rows_done = 0
    reuse = False
    record_counts = {"caption_nonempty": 0, "ocr_nonempty": 0, "parse_failed": 0}
    if RESUME and records_path.is_file() and plan_path.is_file():
        try:
            previous = json.loads(plan_path.read_text(encoding="utf-8"))
            plan_ok = (
                previous.get("plan_hash") == current_hash
                and int(previous.get("count", -1)) == total
            )
        except (OSError, ValueError):
            plan_ok = False

        if plan_ok:
            lines = records_path.read_text(encoding="utf-8").splitlines()
            valid = []
            for index, line in enumerate(lines):
                if index >= total:
                    break
                try:
                    record = json.loads(line)
                except ValueError:
                    break
                if int(record.get("kf_uid", -1)) != uids[index]:
                    break
                valid.append(line)
                if (record.get("caption") or "").strip():
                    record_counts["caption_nonempty"] += 1
                if (record.get("ocr") or "").strip():
                    record_counts["ocr_nonempty"] += 1
                if not record.get("parsed", False):
                    record_counts["parse_failed"] += 1
            rows_done = len(valid)
            reuse = True
            if rows_done != len(lines):
                tmp = records_path.with_suffix(".jsonl.tmp")
                tmp.write_text(
                    "\n".join(valid) + ("\n" if valid else ""), encoding="utf-8"
                )
                os.replace(tmp, records_path)

    if not reuse:
        records_path.write_text("", encoding="utf-8")
        manifest_path.unlink(missing_ok=True)
        plan_path.unlink(missing_ok=True)
        rows_done = 0
        record_counts = {"caption_nonempty": 0, "ocr_nonempty": 0, "parse_failed": 0}

    write_json(
        plan_path,
        {
            "video_id": video_id,
            "model": MODEL_ID,
            "prompt_version": PROMPT_VERSION,
            "image_size": IMAGE_SIZE,
            "max_num": MAX_NUM,
            "max_new_tokens": MAX_NEW_TOKENS,
            "num_beams": NUM_BEAMS,
            "repetition_penalty": REPETITION_PENALTY,
            "count": total,
            "plan_hash": current_hash,
        },
    )

    if reuse and rows_done == total:
        write_json(
            manifest_path,
            shard_manifest(
                video_id,
                total,
                current_hash,
                record_counts["caption_nonempty"],
                record_counts["ocr_nonempty"],
                record_counts["parse_failed"],
            ),
        )
        done_keyframes += total
        print(f"[{position}/{len(selected)}] {video_id}: skip (done)")
        shard_summaries.append({"video_id": video_id, "count": total, "skipped": True})
        continue

    write_json(progress_path, {"rows_done": rows_done, "status": "running"})
    started = time.time()
    print(
        f"[{position}/{len(selected)}] {video_id}: start "
        f"({rows_done:,}/{total:,} keyframes)",
        flush=True,
    )

    def load_batch(start: int) -> list[torch.Tensor]:
        return [load_image(BUNDLE_DIR / row["path"]) for row in video_rows[start : start + BATCH_SIZE]]

    pending = decode_pool.submit(load_batch, rows_done) if rows_done < total else None
    with records_path.open("a", encoding="utf-8") as handle:
        for batch_index, start in enumerate(range(rows_done, total, BATCH_SIZE), start=1):
            tensors = pending.result()
            upcoming = start + BATCH_SIZE
            pending = decode_pool.submit(load_batch, upcoming) if upcoming < total else None

            responses = infer_batch(tensors)
            for offset in range(len(tensors)):
                response = responses[offset] if offset < len(responses) else ""
                record = make_record(video_rows[start + offset], response)
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")
                if record["caption"]:
                    record_counts["caption_nonempty"] += 1
                if record["ocr"]:
                    record_counts["ocr_nonempty"] += 1
                if not record["parsed"]:
                    record_counts["parse_failed"] += 1
            handle.flush()

            if batch_index % PROGRESS_EVERY == 0:
                processed = min(start + len(tensors), total)
                write_json(
                    progress_path,
                    {"rows_done": processed, "status": "running"},
                )
                elapsed = time.time() - overall_started
                so_far = done_keyframes + processed
                rate = so_far / elapsed if elapsed else 0
                remaining = sum(len(by_video[v]) for v in selected) - so_far
                eta_min = remaining / rate / 60 if rate else float("nan")
                print(
                    f"  {video_id}: {processed:,}/{total:,} | "
                    f"{rate:.2f} kf/s | ETA batch {eta_min:,.0f} min",
                    flush=True,
                )

    write_json(progress_path, {"rows_done": total, "status": "done"})
    write_json(
        manifest_path,
        shard_manifest(
            video_id,
            total,
            current_hash,
            record_counts["caption_nonempty"],
            record_counts["ocr_nonempty"],
            record_counts["parse_failed"],
        ),
    )
    done_keyframes += total
    shard_summaries.append({"video_id": video_id, "count": total, "skipped": False})
    print(
        f"[{position}/{len(selected)}] {video_id}: {total:,} keyframes "
        f"in {time.time() - started:,.1f}s",
        flush=True,
    )

decode_pool.shutdown(wait=True)
print(
    f"done {len(shard_summaries):,} videos in "
    f"{(time.time() - overall_started) / 60:,.1f} min",
    flush=True,
)
if not VIDEO_IDS and VIDEO_LIMIT > 0:
    print("next VINTERN_VIDEO_OFFSET:", VIDEO_OFFSET + len(selected), flush=True)


## 9. Đóng gói shard

Zip chỉ chứa `shards/<video_id>/caption_ocr.jsonl` + `manifest.json` của batch này
(không chứa keyframes, không chứa `progress.json`/`plan.json`).


In [ ]:
import zipfile

totals = {"caption_nonempty": 0, "ocr_nonempty": 0, "parse_failed": 0}
total_keyframes = 0
for summary in shard_summaries:
    shard_dir = SHARDS_DIR / summary["video_id"]
    manifest = json.loads((shard_dir / "manifest.json").read_text(encoding="utf-8"))
    total_keyframes += int(manifest["count"])
    for key in totals:
        totals[key] += int(manifest.get(key, 0))

run_manifest = {
    "version": 1,
    "created_at": datetime.now(UTC).isoformat(),
    "model": MODEL_ID,
    "prompt_version": PROMPT_VERSION,
    "image_size": IMAGE_SIZE,
    "max_num": MAX_NUM,
    "use_thumbnail": USE_THUMBNAIL,
    "max_new_tokens": MAX_NEW_TOKENS,
    "num_beams": NUM_BEAMS,
    "repetition_penalty": REPETITION_PENALTY,
    "dtype": DTYPE_NAME,
    "videos": len(shard_summaries),
    "keyframes": total_keyframes,
    **totals,
}

README = f"""AIC2024 KIS - Vintern-1B-v3.5 caption + OCR shards
model: {MODEL_ID}
prompt: {PROMPT_VERSION} | image_size: {IMAGE_SIZE} | max_num: {MAX_NUM}
keyframes: {total_keyframes:,}

Each line of shards/<video_id>/caption_ocr.jsonl:
  kf_uid, video_id, generation, shot_idx, kf_idx, ts_ms,
  caption (detail description), ocr (recognized text or ""), parsed

parsed=false means the model did not return valid JSON; those rows also carry "raw".
"""

ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
written = 0
with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
) as archive:
    for summary in shard_summaries:
        shard_dir = SHARDS_DIR / summary["video_id"]
        for path in sorted(shard_dir.glob("*")):
            if not path.is_file() or path.name in {"progress.json", "plan.json"}:
                continue
            archive.write(
                path,
                arcname=f"shards/{summary['video_id']}/{path.name}",
            )
            written += 1
    archive.writestr("manifest.json", json.dumps(run_manifest, indent=2, ensure_ascii=False))
    archive.writestr("README.txt", README)

print(f"zip: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:,.1f} MB, {written} files)")
print(json.dumps(run_manifest, ensure_ascii=False, indent=2))


## 10. Kiểm tra batch

Verify số dòng `caption_ocr.jsonl` khớp `manifest.count`, in tỉ lệ có caption / có OCR /
parse lỗi và một vài mẫu để soi chất lượng.


In [ ]:
total_keyframes = 0
totals = {"caption_nonempty": 0, "ocr_nonempty": 0, "parse_failed": 0}
for summary in shard_summaries:
    shard_dir = SHARDS_DIR / summary["video_id"]
    manifest = json.loads((shard_dir / "manifest.json").read_text(encoding="utf-8"))
    with (shard_dir / "caption_ocr.jsonl").open(encoding="utf-8") as handle:
        count = sum(1 for line in handle if line.strip())
    assert count == int(manifest["count"]), (
        summary["video_id"],
        count,
        manifest["count"],
    )
    total_keyframes += count
    for key in totals:
        totals[key] += int(manifest.get(key, 0))
    print(f"{summary['video_id']}: {count:,} keyframes")

print(f"batch: {len(shard_summaries):,} videos, {total_keyframes:,} keyframes")
for key, value in totals.items():
    share = value / total_keyframes * 100 if total_keyframes else 0
    print(f"  {key:>16}: {value:,} ({share:.1f}%)")

samples_file = SHARDS_DIR / shard_summaries[0]["video_id"] / "caption_ocr.jsonl"
with samples_file.open(encoding="utf-8") as handle:
    for _, line in zip(range(3), handle):
        record = json.loads(line)
        print("\n", record["video_id"], record["kf_uid"], flush=True)
        print("caption:", record["caption"][:200], flush=True)
        print("ocr:", record["ocr"][:200].replace("\n", " / "), flush=True)


## Sau khi chạy

1. Tải `artifacts_caption_ocr_shards.zip`.
2. Giữ nguyên làm artifact gốc: `caption_ocr.jsonl` dễ đọc, dễ grep, không phụ thuộc SQLite.
3. Các bước sau (chưa làm trong notebook này):
   - import `ocr` vào cột `keyframes.ocr_text` của SQLite để boost search text;
   - build text index (BM25/FAISS text) từ `caption` để search tiếng Việt trực tiếp;
   - filter keyframe theo OCR trùng query (số báo, tên riêng, logo đài).
4. Chạy nhiều batch với `VINTERN_VIDEO_OFFSET` khác nhau; mỗi batch tạo một zip riêng,
   import/resume đều an toàn vì mỗi dòng gắn `kf_uid`.
